In [7]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [8]:
df = pd.read_excel("Oleogel-HSI-average-bread-hardness-data.xlsx", header=0)

df1 = df.drop(['Sample', 'Concentration', 'Gelator', 'Oil'], axis=1)

print(type(df))
print(df.shape)

df

<class 'pandas.core.frame.DataFrame'>
(2997, 229)


,Sample,Concentration,Gelator,Oil,0,1,2,3,4,5,...,215,216,217,218,219,220,221,222,223,Hardness
0,Control,Control,Control,Control,0.195594,0.173271,0.150947,0.128624,0.106300,0.083976,...,0.010554,0.019257,0.028031,0.036621,0.044904,0.053187,0.061470,0.069752,0.078035,7.166
1,Control,Control,Control,Control,0.195779,0.173344,0.150909,0.128473,0.106038,0.083603,...,0.010628,0.019244,0.027728,0.036033,0.043928,0.051822,0.059716,0.067610,0.075504,7.166
2,Control,Control,Control,Control,0.195995,0.173533,0.151071,0.128608,0.106146,0.083683,...,0.011105,0.019679,0.028304,0.036699,0.044727,0.052755,0.060784,0.068812,0.076840,7.166
3,Control,Control,Control,Control,0.195778,0.173358,0.150938,0.128518,0.106098,0.083678,...,0.011309,0.019865,0.028471,0.036615,0.044545,0.052475,0.060406,0.068336,0.076266,7.166
4,Control,Control,Control,Control,0.197401,0.174722,0.152042,0.129363,0.106683,0.084004,...,0.011008,0.019413,0.027794,0.036131,0.044092,0.052053,0.060015,0.067976,0.075937,7.166
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2992,15BF,15,Beeswax,Flaxseed,0.192546,0.170707,0.148869,0.127030,0.105192,0.083353,...,0.010637,0.018995,0.027537,0.035549,0.043242,0.050935,0.058627,0.066320,0.074012,5.466
2993,15BF,15,Beeswax,Flaxseed,0.190679,0.169127,0.147575,0.126023,0.104471,0.082919,...,0.010748,0.019138,0.027336,0.035155,0.042568,0.049981,0.057394,0.064807,0.072220,5.466
2994,15BF,15,Beeswax,Flaxseed,0.193693,0.171661,0.149630,0.127598,0.105566,0.083535,...,0.010497,0.018978,0.026930,0.034767,0.042119,0.049471,0.056823,0.064174,0.071526,5.466
2995,15BF,15,Beeswax,Flaxseed,0.194202,0.172094,0.149985,0.127876,0.105768,0.083659,...,0.010702,0.018749,0.026960,0.034856,0.042459,0.050063,0.057666,0.065270,0.072873,5.466


In [9]:
target_name = "hardness"

df1 = df.drop(['Sample', 'Concentration', 'Gelator', 'Oil'], axis=1)

# Streamlit에서 업로드한 엑셀 파일의 입력 컬럼을 맞추기 위해 저장
feature_columns = df1.columns[:-1].tolist()

data_raw2 = np.array(df1, dtype=np.float32)

x_data = data_raw2[:, 0:-1]
y_data = data_raw2[:, [-1]]

x_train, x_test, y_train, y_test = train_test_split(
    x_data,
    y_data,
    test_size=0.2,
    random_state=42
)

# x 정규화
x_scaler = MinMaxScaler()
x_train_scaled = x_scaler.fit_transform(x_train)
x_test_scaled = x_scaler.transform(x_test)

# y 정규화
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

In [10]:
import joblib
from pathlib import Path

save_dir = Path(".")

scatter_list = []


def add_scatter_data(model_name, y_train_true, y_train_pred, y_test_true, y_test_pred):
    y_train_true = np.array(y_train_true).ravel()
    y_test_true = np.array(y_test_true).ravel()
    y_train_pred = np.array(y_train_pred).ravel()
    y_test_pred = np.array(y_test_pred).ravel()

    for actual, pred in zip(y_train_true, y_train_pred):
        scatter_list.append({
            "Model": model_name,
            "Dataset": "Train",
            "Actual": actual,
            "Predicted": pred
        })

    for actual, pred in zip(y_test_true, y_test_pred):
        scatter_list.append({
            "Model": model_name,
            "Dataset": "Test",
            "Actual": actual,
            "Predicted": pred
        })

In [12]:
results_list = []

def save_result(model_name, y_train_true, y_train_pred, y_test_true, y_test_pred):
    y_train_true = np.array(y_train_true).ravel()
    y_test_true = np.array(y_test_true).ravel()
    y_train_pred = np.array(y_train_pred).ravel()
    y_test_pred = np.array(y_test_pred).ravel()

    R2_train = r2_score(y_train_true, y_train_pred)
    R2_test = r2_score(y_test_true, y_test_pred)

    RMSE_train = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
    RMSE_test = np.sqrt(mean_squared_error(y_test_true, y_test_pred))

    results_list.append({
        "Model": model_name,
        "R2_Train": R2_train,
        "R2_Test": R2_test,
        "RMSE_Train": RMSE_train,
        "RMSE_Test": RMSE_test
    })

    add_scatter_data(
        model_name,
        y_train_true,
        y_train_pred,
        y_test_true,
        y_test_pred
    )

    print("Model:", model_name)
    print("R2 (train):", R2_train)
    print("R2 (test):", R2_test)
    print("RMSE (train):", RMSE_train)
    print("RMSE (test):", RMSE_test)
    print("=" * 30)


models = {
    "RandomForest": RandomForestRegressor(random_state=42),
    "SVM": SVR(kernel="linear", C=10, gamma=1),
    "AdaBoost": AdaBoostRegressor(n_estimators=100, random_state=42),
    "GBM": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    "CatBoost": CatBoostRegressor(n_estimators=100, random_state=42, verbose=0),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "KNeighbors": KNeighborsRegressor(n_neighbors=5),
    "LightGBM": LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
}


# scaler는 모델별로 저장하지 않고 타깃별로 한 번만 저장
joblib.dump(x_scaler, f"x_scaler_{target_name}.pkl")
joblib.dump(y_scaler, f"y_scaler_{target_name}.pkl")


for key, model in models.items():

    model.fit(x_train_scaled, y_train_scaled.ravel())

    y_train_pred = model.predict(x_train_scaled)
    y_test_pred = model.predict(x_test_scaled)

    save_result(
        key,
        y_train_scaled,
        y_train_pred,
        y_test_scaled,
        y_test_pred
    )

    # 모델만 모델별로 저장
    joblib.dump(model, f"{key}_{target_name}_model.pkl")

Model: RandomForest
R2 (train): 0.7602031403485331
R2 (test): 0.6647641051797746
RMSE (train): 0.09818328157377745
RMSE (test): 0.11975682222993693
Model: SVM
R2 (train): 0.6852112232372729
R2 (test): 0.6418671247993125
RMSE (train): 0.11249297898737462
RMSE (test): 0.12377903746444777
Model: AdaBoost
R2 (train): 0.666254613729228
R2 (test): 0.650710947908843
RMSE (train): 0.11583063389689685
RMSE (test): 0.12224116867783477
Model: GBM
R2 (train): 0.7569906432256652
R2 (test): 0.6970635788898614
RMSE (train): 0.0988387616931602
RMSE (test): 0.11384153727696741
Model: XGBoost
R2 (train): 0.7605321407318115
R2 (test): 0.6651706695556641
RMSE (train): 0.09811589786506057
RMSE (test): 0.11968417934338085
Model: CatBoost
R2 (train): 0.7600074796670186
R2 (test): 0.6768561032903693
RMSE (train): 0.09822332940993214
RMSE (test): 0.11757716527462302
Model: DecisionTree
R2 (train): 0.7605341214523975
R2 (test): 0.6643910044934584
RMSE (train): 0.09811549913452819
RMSE (test): 0.1198234453573288

C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [14]:
# 3) MLP 모델 학습 및 저장
tf.keras.backend.clear_session()
tf.random.set_seed(42)

mlp_model = Sequential([
    Input(shape=(x_train_scaled.shape[1],)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1)
])

mlp_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True
)

mlp_model.fit(
    x_train_scaled,
    y_train_scaled,
    validation_split=0.2,
    epochs=500,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

y_train_pred_mlp = mlp_model.predict(x_train_scaled, verbose=0).ravel()
y_test_pred_mlp = mlp_model.predict(x_test_scaled, verbose=0).ravel()

save_result(
    "MLP",
    y_train_scaled,
    y_train_pred_mlp,
    y_test_scaled,
    y_test_pred_mlp
)

# MLP 모델만 저장
mlp_model.save(f"MLP_{target_name}_model.keras")

Model: MLP
R2 (train): 0.7175235748291016
R2 (test): 0.688807487487793
RMSE (train): 0.10656310921715802
RMSE (test): 0.11538240655288713


In [15]:
# ==============================
# CNN 모델 학습 및 저장
# ==============================

tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_features = x_train_scaled.shape[1]

cnn_model = Sequential([
    Input(shape=(n_features, 1)),
    Conv1D(filters=32, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(1)
])

cnn_model.compile(
    optimizer="adam",
    loss="mse"
)

# CNN 입력 형태로 변환
x_train_cnn = x_train_scaled.reshape(
    x_train_scaled.shape[0],
    x_train_scaled.shape[1],
    1
)

x_test_cnn = x_test_scaled.reshape(
    x_test_scaled.shape[0],
    x_test_scaled.shape[1],
    1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

cnn_model.fit(
    x_train_cnn,
    y_train_scaled,
    validation_split=0.2,
    epochs=100,
    batch_size=8,
    callbacks=[early_stop],
    verbose=0
)

# 예측
y_train_pred_cnn = cnn_model.predict(x_train_cnn, verbose=0).ravel()
y_test_pred_cnn = cnn_model.predict(x_test_cnn, verbose=0).ravel()

# 결과 저장
save_result(
    "CNN",
    y_train_scaled,
    y_train_pred_cnn,
    y_test_scaled,
    y_test_pred_cnn
)

# CNN 모델만 저장
cnn_model.save(f"CNN_{target_name}_model.keras")

Model: CNN
R2 (train): 0.708639919757843
R2 (test): 0.6946626901626587
RMSE (train): 0.10822580096473171
RMSE (test): 0.11429177293060949


In [29]:
# 최종 성능 결과 저장
results_df = pd.DataFrame(results_list)
scatter_df = pd.DataFrame(scatter_list)

results_df.to_csv(f"{target_name}_performance.csv", index=False)
scatter_df.to_csv(f"{target_name}_scatter_data.csv", index=False)

# 입력 feature column 저장
joblib.dump(feature_columns, f"feature_columns_{target_name}.pkl")

# 공통 scaler도 추가 저장해두면 좋음
joblib.dump(x_scaler, f"x_scaler_{target_name}.pkl")
joblib.dump(y_scaler, f"y_scaler_{target_name}.pkl")

print("저장 완료")
print(f"{target_name}_performance.csv")
print(f"{target_name}_scatter_data.csv")
print(f"feature_columns_{target_name}.pkl")
print(f"x_scaler_{target_name}.pkl")
print(f"y_scaler_{target_name}.pkl")

results_df

저장 완료
hardness_performance.csv
hardness_scatter_data.csv
feature_columns_hardness.pkl
x_scaler_hardness.pkl
y_scaler_hardness.pkl


,Model,R2_Train,R2_Test,RMSE_Train,RMSE_Test
0,RandomForest,0.760203,0.664757,0.098183,0.119758
1,SVM,0.685211,0.641867,0.112493,0.123779
2,AdaBoost,0.666255,0.650711,0.115831,0.122241
3,GBM,0.756991,0.697064,0.098839,0.113842
4,XGBoost,0.760532,0.665171,0.098116,0.119684
5,CatBoost,0.760007,0.676856,0.098223,0.117577
6,DecisionTree,0.760534,0.664391,0.098115,0.119823
7,KNeighbors,0.739826,0.641335,0.102270,0.123871
8,LightGBM,0.760521,0.666722,0.098118,0.119407
9,MLP,0.721315,0.703493,0.105846,0.112627


In [16]:
from pathlib import Path

check_files = [
    "hardness_performance.csv",
    "hardness_scatter_data.csv",
    "feature_columns_hardness.pkl",
    "x_scaler_hardness.pkl",
    "y_scaler_hardness.pkl"
]

print("=" * 50)
print("파일 생성 확인")
print("=" * 50)

for file in check_files:
    if Path(file).exists():
        print("OK  :", file)
    else:
        print("MISS:", file)

파일 생성 확인
OK  : hardness_performance.csv
OK  : hardness_scatter_data.csv
MISS: feature_columns_hardness.pkl
OK  : x_scaler_hardness.pkl
OK  : y_scaler_hardness.pkl
